### INSTALL DEPS FOR TRAINING

In [ ]:
!pip install  transformers datasets sacrebleu sentencepiece accelerate evaluate


## LOADING THE DATASET

In [ ]:
from datasets import load_dataset
raw = load_dataset("csv", data_files="/content/twi_translation.csv")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
df = raw['train'].to_pandas().rename(columns={'text': 'twi', 'label': 'english'})
# Drop rows where either column is missing
df = df.dropna(subset=['twi', 'english'])

print(f'Loaded {len(df):,} rows')
df.head(5)

Loaded 10,409 rows


,id,twi,english
0,44224,"Ampa ara, bosome Ɔbenem da a ɛtɔ so nsia wɔ af...","Indeed, the sixth day of the month of March 19..."
1,43602,Wobɛtumi nso de aduro a wɔayɛ sɛ yɛmfa nhohoro...,You can also rinse your mouth with some of the...
2,41752,Berɛ a wɔretoto Owura Charles Abani ano wɔ Joy...,During an interview with Mr. Charles Abani on ...
3,47022,"Yei nti, ama nnipa pii ani abɛgye nwa ho pa ara.",This has led to a growing interest in drinking.
4,42093,"Deɛ ɛtwa toɔ no, sɛ wotaa di aduaba yi a, ɛbɛb...","Finally, regular consumption of this fruit wil..."


Based on inspection of the sample data, we apply these quality filters:
- Drop rows where either side is empty or NaN
- Drop rows shorter than 4 tokens (too short for meaningful translation)
- Drop rows longer than 200 tokens (outliers that blow up GPU memory)
- Strip leading numbers like `2.The coconut...` (artefacts from the source scraping)
- Deduplicate exact pairs

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return None
    # Remove leading list numbers like "2." or "3. "
    text = re.sub(r'^\d+\.\s*', '', text.strip())
    # Collapse multiple spaces
    text = re.sub(r' +', ' ', text)
    return text.strip() if text.strip() else None

df['twi']     = df['twi'].apply(clean_text)
df['english'] = df['english'].apply(clean_text)

# Drop nulls introduced by cleaning
df = df.dropna(subset=['twi', 'english'])

# Length filter (word count as a proxy for token count)
df['twi_len'] = df['twi'].str.split().str.len()
df['eng_len'] = df['english'].str.split().str.len()
df = df[(df['twi_len'] >= 4) & (df['eng_len'] >= 4)]
df = df[(df['twi_len'] <= 200) & (df['eng_len'] <= 200)]

# Deduplicate
df = df.drop_duplicates(subset=['twi', 'english'])
df = df.drop(columns=['twi_len', 'eng_len'])

print(f'After cleaning: {len(df):,} rows')
df.sample(3)

After cleaning: 10,370 rows


,id,twi,english
983,42822,"Wɔfrɛɛ saa ntoma no ""Nwen-ntoma"" ɛsiane sɛ wɔa...",They named that cloth ''Weaved-cloth'' because...
3865,46858,Yei nso tumi ma asase no ahoɔden so te koraa.,"This, in turn, can greatly reduce the strength..."
1683,43865,"Saa asasewosoɔ yi kunkum nnipa bɛboro 75, 000.","This earthquake killed more than 75,000 people."


In [ ]:
!pip install -q scikit-learn

## TRAIN VAL TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split


train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')

Train: 8,296  |  Val: 1,037  |  Test: 1,037


## 5. Load model & tokenizer

We use `Helsinki-NLP/opus-mt-mul-en` (many languages → English).  
For **English → Twi**, swap in `opus-mt-en-mul` below.

> **Why Helsinki-NLP?** It's small (~300MB), fast to fine-tune, and already knows multilingual structure. NLLB-200 is more powerful but needs ~5GB RAM — not ideal for a free T4.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# Twi → English  (change to 'Helsinki-NLP/opus-mt-en-mul' for En→Twi)
MODEL_NAME = 'Helsinki-NLP/opus-mt-mul-en'

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model     = MarianMTModel.from_pretrained(MODEL_NAME)

print('Vocab size:', tokenizer.vocab_size)
print('Model params:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/707k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/791k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/310M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Vocab size: 64172
Model params: 143.230976 M


## 6. Tokenize

MarianMT expects the source in `input_ids` and the target as `labels`.

In [ ]:
from datasets import Dataset

MAX_LEN = 256  # tokens — covers ~95% of sentences in this dataset

def tokenize(batch):
    model_inputs = tokenizer(
        batch['twi'],
        max_length=MAX_LEN,
        truncation=True,
        padding='max_length'
    )
    # Removed with tokenizer.as_target_tokenizer() as it's causing an AttributeError.
    # The tokenizer is expected to handle target language tokenization directly.
    labels = tokenizer(
        batch['english'],
        max_length=MAX_LEN,
        truncation=True,
        padding='max_length'
    )
    # Replace padding token id in labels with -100 so loss ignores them
    labels['input_ids'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Convert to HuggingFace Dataset objects
train_ds = Dataset.from_pandas(train_df[['twi','english']].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[['twi','english']].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df[['twi','english']].reset_index(drop=True))

train_tok = train_ds.map(tokenize, batched=True, batch_size=256, remove_columns=['twi','english'])
val_tok   = val_ds.map(tokenize,   batched=True, batch_size=256, remove_columns=['twi','english'])
test_tok  = test_ds.map(tokenize,  batched=True, batch_size=256, remove_columns=['twi','english'])

print('Tokenization complete.')
print('Train sample keys:', train_tok[0].keys())

Map:   0%|          | 0/8296 [00:00<?, ? examples/s]

Map:   0%|          | 0/1037 [00:00<?, ? examples/s]

Map:   0%|          | 0/1037 [00:00<?, ? examples/s]

Tokenization complete.
Train sample keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


## 7. Define evaluation metrics (BLEU + chrF)

In [ ]:
import evaluate
import numpy as np

bleu_metric = evaluate.load('sacrebleu')
chrf_metric = evaluate.load('chrf')

def compute_metrics(eval_preds):
    preds, labels = eval_preds


    # Decode predictions
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.argmax(preds, axis=-1) if preds.ndim == 3 else preds
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Decode labels (replace -100 with pad token id first)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # sacrebleu expects list-of-strings predictions, list-of-list-of-strings references
    bleu_result = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[ref] for ref in decoded_labels]
    )
    chrf_result = chrf_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    return {
        'bleu': round(bleu_result['score'], 2),
        'chrf': round(chrf_result['score'], 2)
    }

print('Metrics ready.')

Metrics ready.


## 8. Fine-tune

Training ~6,000 pairs for 5 epochs on a T4 takes roughly **20–35 minutes**.

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir='./twi-en-marianmt',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=5e-5,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='bleu',
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to='none'
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,3.251555,3.004178,4.320000,19.850000
2,2.682213,2.593195,5.420000,23.280000
3,2.385563,2.399730,6.980000,26.990000
4,2.139953,2.301383,8.370000,29.210000
5,2.049161,2.274319,8.800000,30.050000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=2595, training_loss=2.9378420379579873, metrics={'train_runtime': 1534.1669, 'train_samples_per_second': 27.037, 'train_steps_per_second': 1.691, 'total_flos': 2812208354426880.0, 'train_loss': 2.9378420379579873, 'epoch': 5.0})

## 9. Evaluate on the held-out test set

In [ ]:
results = trainer.evaluate(test_tok)
print(f"Test BLEU : {results.get('eval_bleu', 'N/A')}")
print(f"Test chrF : {results.get('eval_chrf', 'N/A')}")

# ── Score interpretation guide ────────────────────────────────────────────────
# BLEU < 10  → Poor, needs more data or model tuning
# BLEU 10-20 → Decent baseline for a low-resource language
# BLEU 20-30 → Good — usable in downstream apps
# BLEU > 30  → Excellent for a low-resource language like Twi
#
# chrF is often more reliable than BLEU for morphologically rich languages.
# chrF > 40 is a solid result for Twi.

Test BLEU : 8.38
Test chrF : 30.52


## 10. Quick inference demo

In [ ]:
def translate(twi_text: str) -> str:
    inputs = tokenizer([twi_text], return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    translated = model.generate(**inputs, num_beams=4, max_length=MAX_LEN)
    return tokenizer.decode(translated[0], skip_special_tokens=True)

# Test on a few examples from the original dataset
test_sentences = [
    'Na mmofra no yɛ sukuufo a wɔwɔ asɛmpatrɛw sukuu ahorow mu',
    'Ɛsɛ sɛ ɔmanfo hu sɛnea wobetumi akura nipadua a ɛte apɔw mu.',
    'Polisifo dwumadie ne sɛ wɔbɛbɔ nnipa a wɔwɔ ɔman no mu ho ban na wɔasom wɔn.'
]

for twi in test_sentences:
    print(f'Twi    : {twi}')
    print(f'English: {translate(twi)}')
    print()

Twi    : Na mmofra no yɛ sukuufo a wɔwɔ asɛmpatrɛw sukuu ahorow mu
English: High children are schoolers who have access to earthquakes.

Twi    : Ɛsɛ sɛ ɔmanfo hu sɛnea wobetumi akura nipadua a ɛte apɔw mu.
English: Additionally, the country should try to prevent a healthy body.

Twi    : Polisifo dwumadie ne sɛ wɔbɛbɔ nnipa a wɔwɔ ɔman no mu ho ban na wɔasom wɔn.
English: Police is trying to encourage people in the country and to encourage them.



### RESUME TRAINING WITH 2595 BUT INCREASE EPOCHS TO 10

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Point to your best checkpoint
CHECKPOINT = './twi-en-marianmt/checkpoint-2595'

# Load model and tokenizer FROM the checkpoint
from transformers import MarianMTModel, MarianTokenizer
tokenizer = MarianTokenizer.from_pretrained(CHECKPOINT)
model     = MarianMTModel.from_pretrained(CHECKPOINT)

# New training args — 10 total epochs (continues from where epoch 5 left off)
training_args = Seq2SeqTrainingArguments(
    output_dir='./twi-en-marianmt-continued',
    num_train_epochs=10,          # 5 new epochs on top of the 5 already done
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,              # lower warmup since we're continuing, not starting fresh
    weight_decay=0.01,
    learning_rate=2e-5,           # lower LR than before (was 5e-5) — avoids overshooting
    predict_with_generate=True,
    generation_max_length=128,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='bleu',
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to='none'
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,      # same tokenized datasets from before
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Resume from checkpoint
trainer.train(resume_from_checkpoint=CHECKPOINT)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss,Bleu,Chrf
6,1.988178,2.220728,9.370000,30.410000
7,1.868894,2.155855,9.970000,32.030000
8,1.743589,2.124636,10.660000,33.020000
9,1.702450,2.104321,11.060000,33.520000
10,1.655463,2.098516,11.040000,33.780000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=5190, training_loss=0.9075643212349658, metrics={'train_runtime': 1419.2043, 'train_samples_per_second': 58.455, 'train_steps_per_second': 3.657, 'total_flos': 5624416708853760.0, 'train_loss': 0.9075643212349658, 'epoch': 10.0})

In [ ]:
# Manually test the final model to get a true picture
def evaluate_manually(model, tokenizer, test_df, n=100):
    import sacrebleu
    preds, refs = [], []
    sample = test_df.sample(n, random_state=42)

    for _, row in sample.iterrows():
        inputs = tokenizer([row['twi']], return_tensors='pt',
                          padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        out = model.generate(**inputs, num_beams=4, max_length=128)
        pred = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(pred)
        refs.append(row['english'])

    bleu = sacrebleu.corpus_bleu(preds, [refs])
    chrf = sacrebleu.corpus_chrf(preds, [refs])
    print(f"BLEU : {bleu.score:.2f}")
    print(f"chrF : {chrf.score:.2f}")

    # Print some examples
    for twi, pred, ref in zip(sample['twi'].values[:5], preds[:5], refs[:5]):
        print(f"\nTwi  : {twi}")
        print(f"Pred : {pred}")
        print(f"Ref  : {ref}")

evaluate_manually(model, tokenizer, test_df)

BLEU : 10.47
chrF : 33.30

Twi  : Confusion within supporters rose when two brothers contested for the same post and area.
Pred : Nnipa a wɔde wɔn ho hyɛ wɔn a wɔte hɔ no mu bere a na wɔde wɔn ho hyɛɛɛ ne mpɔtam hɔ no mu.
Ref  : Basabasayɛ baa akyigyinafo mu bere a anuanom baanu sii akan wɔ dibea ne beae koro no ara ho no.

Twi  : Some government projects take so long to be implemented.
Pred : Aban nhyehyɛe ahodoɔ bi a ɛbɛboa ama wɔatumi ayɛ adwuma.
Ref  : Aban nnwuma bi gye bere tenten saa ansa na wɔde adi dwuma.

Twi  : Money extortion issues are very common these days.
Pred : Sika a wɔde sika ahorow ho nsɛm ho nsɛm pii wɔ nnansa yi mu.
Ref  : Sika a wɔde ahoɔden anaa ahunahuna gye ho nsɛm abu so kɛse nnansa yi.

Twi  : Ɛboa ma wo so tumi te.
Pred : It helps to improve you.
Ref  : It help reduce weight.

Twi  : Wohwɛ ɛfiri berɛ a anibue baeɛ de bɛsi ɛnnɛ a, wobɛhunu sɛ nsakraeɛ bebree na aba nnipa asetenam.
Pred : When you look at the rainy days today, you will know that many culture

In [ ]:
# Detect direction by checking if input looks like English
# (simple heuristic: English uses only ASCII, Twi has special chars like ɛ ɔ ŋ)
def is_english(text):
    return all(ord(c) < 256 for c in text.replace(' ', ''))

test_df['direction'] = test_df['twi'].apply(
    lambda x: 'en_to_twi' if is_english(x) else 'twi_to_en'
)

print(test_df['direction'].value_counts())

# Evaluate each direction separately
for direction in ['twi_to_en', 'en_to_twi']:
    subset = test_df[test_df['direction'] == direction]
    print(f"\n--- {direction} ({len(subset)} pairs) ---")
    evaluate_manually(model, tokenizer, subset, n=min(100, len(subset)))

direction
en_to_twi    580
twi_to_en    457
Name: count, dtype: int64

--- twi_to_en (457 pairs) ---
BLEU : 11.57
chrF : 34.32

Twi  : Owu pa ne sɛ onipa bi ato ne kɔn awu.
Pred : A good child is that a person has a good diet.
Ref  : A good death is when someone dies at old age.

Twi  : Psychological abuse: involves causing fear by intimidation; threatening physical harm to self, partner or children; destruction of pets and property; “mind games”; or forcing isolation from friends, family, school and/or work.
Pred : Mpɔtam hɔfoɔ: ɛsɛ sɛ wɔde wɔn ho hyɛ nhwehwɛmu a, wɔde ma wɔn ho hyɛ nneɛma a wɔde ma wɔn a wɔde ma wɔn ho hyɛ no, partner anaa mmofra, mmarima ne mmarima a wɔde ma wɔn adwene mu no mu; ɛsɛ sɛ "mind ahodoɔ mu"; nnipa a wɔde wɔn ho hyɛ
Ref  : adwene mu ayayade: nea ɛka ho ne sɛ wɔde ehu bɛba denam ahunahuna so; honam fam opira a wɔde hunahuna wɔn ho, ɔhokafo anaa mmofra; afieboa ne agyapade a wɔsɛe no; “adwene mu agodie”; anaasɛ wɔhyɛ obi ma ɔtew ne ho fi nnamfo, abusua, suk

In [ ]:
results = trainer.evaluate(test_tok)
print(f"Test BLEU : {results.get('eval_bleu', 'N/A')}")
print(f"Test chrF : {results.get('eval_chrf', 'N/A')}")

# ── Score interpretation guide ────────────────────────────────────────────────
# BLEU < 10  → Poor, needs more data or model tuning
# BLEU 10-20 → Decent baseline for a low-resource language
# BLEU 20-30 → Good — usable in downstream apps
# BLEU > 30  → Excellent for a low-resource language like Twi
#
# chrF is often more reliable than BLEU for morphologically rich languages.
# chrF > 40 is a solid result for Twi.

## 11. Save locally & push to Hugging Face Hub

Create a free account at https://huggingface.co and get your token from Settings > Access Tokens.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('twi_en_model', 'zip', './twi-en-final')
files.download('twi_en_model.zip')

# Push model to the hub

In [ ]:
# ── Step 1: Login to Hugging Face ─────────────────────────────────────────────
from huggingface_hub import notebook_login
notebook_login()
# A widget will appear — paste your HF token from:
# https://huggingface.co/settings/tokens
# Make sure the token has WRITE permission

In [ ]:
# ── Step 2: Load checkpoint and push ──────────────────────────────────────────
from transformers import MarianMTModel, MarianTokenizer

CHECKPOINT = '/content/twi-en-marianmt/checkpoint-2595'
HF_REPO    = 'ninte/twi-en-marianmt'  # ← replace with your HF username

# Load from checkpoint
model     = MarianMTModel.from_pretrained(CHECKPOINT)
tokenizer = MarianTokenizer.from_pretrained(CHECKPOINT)

# Add a proper model card config before pushing
model.config.update({
    "model_type": "marian",
    "src_lang": "tw",       # Twi
    "tgt_lang": "en",       # English
    "task": "translation",
})

# Push both model and tokenizer
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"Done! View your model at: https://huggingface.co/{HF_REPO}")

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpmtd9lvku/source.spm : 100%|##########|  707kB /  707kB            

  /tmp/tmpmtd9lvku/target.spm : 100%|##########|  791kB /  791kB            

Done! View your model at: https://huggingface.co/ninte/twi-en-marianmt


In [ ]:
# ── Step 1: Load checkpoint ───────────────────────────────────────────────────
from transformers import MarianMTModel, MarianTokenizer
import sacrebleu
import numpy as np

CHECKPOINT = '/content/twi-en-marianmt/checkpoint-2595'

model     = MarianMTModel.from_pretrained(CHECKPOINT)
tokenizer = MarianTokenizer.from_pretrained(CHECKPOINT)
model.eval()

# Move to GPU if available
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f"Running on: {device}")

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Running on: cuda


In [ ]:
# ── Step 2: Full evaluation on test set ───────────────────────────────────────
def full_evaluate(model, tokenizer, df, n=None, label=''):
    sample = df if n is None else df.sample(min(n, len(df)), random_state=42)
    preds, refs = [], []

    for _, row in sample.iterrows():
        inputs = tokenizer(
            [row['twi']], return_tensors='pt',
            padding=True, truncation=True, max_length=128
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(**inputs, num_beams=4, max_length=128)
        pred = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(pred)
        refs.append(row['english'])

    bleu    = sacrebleu.corpus_bleu(preds, [refs])
    chrf    = sacrebleu.corpus_chrf(preds, [refs])
    ter     = sacrebleu.corpus_ter(preds, [refs])

    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  Pairs evaluated : {len(sample)}")
    print(f"  BLEU            : {bleu.score:.2f}")
    print(f"  chrF            : {chrf.score:.2f}")
    print(f"  TER             : {ter.score:.2f}  (lower is better)")
    print(f"  BLEU (1-gram)   : {bleu.precisions[0]:.2f}")
    print(f"  BLEU (2-gram)   : {bleu.precisions[1]:.2f}")
    print(f"  BLEU (3-gram)   : {bleu.precisions[2]:.2f}")
    print(f"  BLEU (4-gram)   : {bleu.precisions[3]:.2f}")
    print(f"  BP (brevity)    : {bleu.bp:.4f}")
    print(f"{'='*50}\n")

    return {
        'bleu': round(bleu.score, 2),
        'chrf': round(chrf.score, 2),
        'ter':  round(ter.score, 2),
        'bp':   round(bleu.score, 4),
        'n':    len(sample),
        'preds': preds,
        'refs':  refs
    }

# Evaluate on full test set
results_all = full_evaluate(model, tokenizer, test_df, label='Full test set')

# Evaluate each direction separately
TWI_CHARS = set('ɛɔŋɛɔŋƐƆŊ')
def is_twi(text):
    return any(c in TWI_CHARS for c in str(text))

twi_to_en_test = test_df[test_df['twi'].apply(is_twi)]
en_to_twi_test = test_df[~test_df['twi'].apply(is_twi)]

results_twi_en = full_evaluate(model, tokenizer, twi_to_en_test, label='Twi → English only')
results_en_twi = full_evaluate(model, tokenizer, en_to_twi_test, label='English → Twi only')


  Full test set
  Pairs evaluated : 1037
  BLEU            : 8.26
  chrF            : 30.36
  TER             : 89.90  (lower is better)
  BLEU (1-gram)   : 36.71
  BLEU (2-gram)   : 11.50
  BLEU (3-gram)   : 4.90
  BLEU (4-gram)   : 2.25
  BP (brevity)    : 1.0000


  Twi → English only
  Pairs evaluated : 384
  BLEU            : 10.50
  chrF            : 32.30
  TER             : 87.64  (lower is better)
  BLEU (1-gram)   : 39.32
  BLEU (2-gram)   : 14.07
  BLEU (3-gram)   : 6.61
  BLEU (4-gram)   : 3.33
  BP (brevity)    : 1.0000


  English → Twi only
  Pairs evaluated : 653
  BLEU            : 6.32
  chrF            : 28.74
  TER             : 91.55  (lower is better)
  BLEU (1-gram)   : 34.70
  BLEU (2-gram)   : 9.48
  BLEU (3-gram)   : 3.53
  BLEU (4-gram)   : 1.37
  BP (brevity)    : 1.0000



In [ ]:
# ── Step 3: Print 8 translation examples (4 good, 4 bad) ─────────────────────
import random

def show_examples(preds, refs, source_texts, n=8):
    # Score each prediction individually
    scored = []
    for pred, ref, src in zip(preds, refs, source_texts):
        s = sacrebleu.sentence_bleu(pred, [ref]).score
        scored.append((s, src, pred, ref))

    scored.sort(key=lambda x: x[0])

    bottom4 = scored[:4]   # worst
    top4    = scored[-4:]  # best

    print("\n── 4 Best translations ──────────────────────────────")
    for score, src, pred, ref in reversed(top4):
        print(f"  Source : {src}")
        print(f"  Pred   : {pred}")
        print(f"  Ref    : {ref}")
        print(f"  BLEU   : {score:.1f}\n")

    print("── 4 Worst translations ─────────────────────────────")
    for score, src, pred, ref in bottom4:
        print(f"  Source : {src}")
        print(f"  Pred   : {pred}")
        print(f"  Ref    : {ref}")
        print(f"  BLEU   : {score:.1f}\n")

show_examples(
    results_all['preds'],
    results_all['refs'],
    test_df.sample(len(results_all['preds']), random_state=42)['twi'].tolist()
)


── 4 Best translations ──────────────────────────────
  Source : Ne dwonsɔ ani tumi sesa.
  Pred   : Their response is "Obiri-Nana (Obiri-na)" or "Oburu".
  Ref    : Their response is "Obiri- Nana (Obiri-na)" or "Oburu".
  BLEU   : 72.9

  Source : If they did not resist your opinion, it means they accepted it.
  Pred   : For this reason, when he was born in Ghana in 1927, he became a lawyer.
  Ref    : For this reason, when he returned to Ghana in 1927, he became a self-employed lawyer.
  BLEU   : 62.7

  Source : Tete no, ɛsiane sɛ na ɛyɛ mmusuo sɛ ɔbaa biara ne ɔbarima bɛda berɛ a wɔngoroo no bra nti, na sɛ ɔbaa bi yɛ saa a, wɔyɛ no Kyiribra.
  Pred   : Some of these are as following;
  Ref    : Some of these are as follows:
  BLEU   : 61.5

  Source : Wɔsusu sɛ Nana Agradaa abɔ n'asɔre mma no apoo, berɛ a ɔsee wɔn sɛ wɔmfa wɔn sika a wɔwɔ no mmra na ɔnyɛ no mmɔho mma wɔn, na ɔgyee wɔn sika nanso wamfa biribiara amma wɔn bio no.
  Pred   : Experts of high blood pressure and diabete

In [ ]:
# ── Step 4: Training stats summary ───────────────────────────────────────────
print("\n── Training summary ─────────────────────────────────")
print(f"  Base model      : Helsinki-NLP/opus-mt-mul-en")
print(f"  Dataset         : GhanaNLP English-Twi Parallel")
print(f"  Total pairs     : {len(df):,}")
print(f"  Train pairs     : {len(train_df):,}")
print(f"  Val pairs       : {len(val_df):,}")
print(f"  Test pairs      : {len(test_df):,}")
print(f"  Epochs          : 5")
print(f"  Batch size      : 16")
print(f"  Learning rate   : 5e-5")
print(f"  Max seq length  : 128")
print(f"  Training time   : 25.6 min (T4 GPU)")
print(f"  Global steps    : 2,595")
print(f"  Final train loss: 2.94")


── Training summary ─────────────────────────────────
  Base model      : Helsinki-NLP/opus-mt-mul-en
  Dataset         : GhanaNLP English-Twi Parallel
  Total pairs     : 10,370
  Train pairs     : 8,296
  Val pairs       : 1,037
  Test pairs      : 1,037
  Epochs          : 5
  Batch size      : 16
  Learning rate   : 5e-5
  Max seq length  : 128
  Training time   : 25.6 min (T4 GPU)
  Global steps    : 2,595
  Final train loss: 2.94
